# Task 3: Granulometry Benchmarking — Qwen2.5-VL-3B (Base Model)

Benchmark the un-fine-tuned model on 108 test images of concrete aggregate.
The model must classify each image by max particle size (8/16/32mm) and grading (coarse/medium/fine).

Two modes:
- **Zero-shot**: No reference — model guesses grading from visual appearance alone
- **Few-shot**: Reference image (`examples_classification_data.png`) included so the model learns A=coarse, B=medium, C=fine

This establishes the baseline that Task 4 (LoRA fine-tuning) should improve.

In [ ]:
import os, json, re, time, torch
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Config

In [ ]:
TEST_DIR = '../../datasets/granulometry/test'
MANIFEST = '../../datasets/granulometry/test_manifest.json'
REF_IMAGE_PATH = 'examples_classification_data.png'
IMG_SIZE = 500  # for preview thumbnails only
ORIGINAL_GSD = 8.0   # pixels per mm at original resolution (2200x3000)
MAX_DIM = 1500        # max pixels on longest side

PROMPT_ZERO_SHOT = '''This is a top-down photograph of concrete aggregate particles.
The ground sampling distance (GSD) is {gsd:.1f} pixels per mm — use this to measure particle sizes.

Classification rules:
- max_particle_size_mm: measure the largest particle visible. Round to the nearest standard sieve size: 8, 16, or 32 mm.
- grading:
  - "coarse" means the majority of particles are larger than 16 mm
  - "medium" means the majority of particles are between 8 mm and 16 mm
  - "fine" means the majority of particles are smaller than 8 mm

Respond with ONLY a JSON object (no other text):
{{"max_particle_size_mm": <8, 16, or 32>, "grading": "<coarse, medium, or fine>"}}'''

PROMPT_FEW_SHOT_REF = '''The first image is a reference chart for classifying concrete aggregate particles.
It is a 3x3 grid of example photographs with labeled rows and columns:
- The 3 COLUMNS represent max particle size, labeled left to right: 8 mm, 16 mm, 32 mm
- The 3 ROWS represent grading type, labeled top to bottom: A (coarse), B (medium), C (fine)
- Each cell shows a real example of that combination (e.g. top-right = A/32mm = coarse grading with 32 mm max particle size)

Grading definitions:
- Coarse (row A): majority of particles are larger than 16 mm
- Medium (row B): majority of particles are between 8 mm and 16 mm
- Fine (row C): majority of particles are smaller than 8 mm

Study this grid carefully, then classify the second image.'''

PROMPT_FEW_SHOT_QUERY = '''Now classify this photograph of concrete aggregate particles.
The ground sampling distance (GSD) is {gsd:.1f} pixels per mm — use this to measure particle sizes.

Compare the particle sizes and distribution to the reference grid.
Which row (A=coarse, B=medium, C=fine) and column (8mm, 16mm, 32mm) does it best match?

Respond with ONLY a JSON object (no other text):
{{"max_particle_size_mm": <8, 16, or 32>, "grading": "<coarse, medium, or fine>"}}'''

## Load Dataset

In [ ]:
with open(MANIFEST) as f:
    manifest = json.load(f)

print(f'Total test images: {len(manifest)}')
print(f'\nClass distribution:')
cls_counts = Counter(e['class'] for e in manifest)
for cls in sorted(cls_counts):
    print(f'  {cls}: {cls_counts[cls]} images')

In [ ]:
classes = sorted(set(e['class'] for e in manifest))
fig, axes = plt.subplots(1, len(classes), figsize=(3*len(classes), 3))
for i, cls in enumerate(classes):
    entry = next(e for e in manifest if e['class'] == cls)
    img = Image.open(os.path.join(TEST_DIR, entry['image'])).convert('RGB')
    img.thumbnail((IMG_SIZE, IMG_SIZE))
    axes[i].imshow(img); axes[i].set_title(f"{cls}\n{entry['grading']}, {entry['max_particle_size_mm']}mm", fontsize=9); axes[i].axis('off')
plt.suptitle('Sample from each class', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Load reference image for few-shot mode — add white background for visibility
ref_image = None
if os.path.exists(REF_IMAGE_PATH):
    raw_ref = Image.open(REF_IMAGE_PATH).convert('RGBA')
    # Composite onto white background so labels (A/B/C, 8/16/32) are visible
    white_bg = Image.new('RGBA', raw_ref.size, (255, 255, 255, 255))
    ref_image = Image.alpha_composite(white_bg, raw_ref).convert('RGB')
    print(f'Reference image loaded: {REF_IMAGE_PATH} ({ref_image.size}) — white background applied')
    plt.figure(figsize=(10, 8))
    plt.imshow(ref_image); plt.title('Reference: Classification Examples (white bg)'); plt.axis('off'); plt.show()
else:
    print(f'WARNING: {REF_IMAGE_PATH} not found — few-shot mode will not be available')

## Load Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')

# Split model across both GPUs, keep GPU 0 lighter for inference tensors
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-3B-Instruct', torch_dtype=torch.bfloat16,
    device_map='auto',
    max_memory={0: '8GiB', 1: '15GiB'},
)
print(f'Model loaded. Device map: {model.hf_device_map}')
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    print(f'  GPU {i}: {alloc:.1f} GB allocated')

## Helper Functions

In [ ]:
def parse_response(raw):
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    size_m = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    grad_m = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if size_m and grad_m:
        return {'max_particle_size_mm': int(size_m.group(1)), 'grading': grad_m.group(1)}
    return None

def infer(img_path, mode='zero-shot', ref_img=None):
    """Load image, resize to MAX_DIM preserving aspect ratio, compute actual GSD, run inference, free memory."""
    image = Image.open(img_path).convert('RGB')
    orig_max = max(image.size)
    scale = min(MAX_DIM / orig_max, 1.0)
    if scale < 1.0:
        new_size = (int(image.width * scale), int(image.height * scale))
        image = image.resize(new_size, Image.LANCZOS)
    actual_gsd = ORIGINAL_GSD * scale
    if mode == 'few-shot' and ref_img is not None:
        msgs = [{'role':'user','content':[
            {'type':'image','image':ref_img},
            {'type':'text','text':PROMPT_FEW_SHOT_REF},
            {'type':'image','image':image},
            {'type':'text','text':PROMPT_FEW_SHOT_QUERY.format(gsd=actual_gsd)},
        ]}]
        images = [ref_img, image]
    else:
        msgs = [{'role':'user','content':[
            {'type':'image','image':image},
            {'type':'text','text':PROMPT_ZERO_SHOT.format(gsd=actual_gsd)},
        ]}]
        images = [image]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=images, return_tensors='pt', padding=True).to(model.device)
    t = time.time()
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    elapsed = time.time() - t
    out = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    del inputs, ids
    image.close()
    torch.cuda.empty_cache()
    return out.strip(), elapsed

def run_benchmark(manifest, mode, ref_img=None):
    results = []
    correct_size = 0
    correct_grading = 0
    valid_json = 0
    total_time = 0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        raw, elapsed = infer(img_path, mode=mode, ref_img=ref_img)
        total_time += elapsed
        parsed = parse_response(raw)
        gt_size = entry['max_particle_size_mm']
        gt_grading = entry['grading']
        size_ok = False
        grading_ok = False
        is_valid = parsed is not None
        if is_valid:
            valid_json += 1
            if parsed.get('max_particle_size_mm') == gt_size: size_ok = True; correct_size += 1
            if parsed.get('grading', '').lower() == gt_grading: grading_ok = True; correct_grading += 1
        results.append({
            'image': entry['image'], 'class': entry['class'],
            'gt_size': gt_size, 'gt_grading': gt_grading,
            'predicted': parsed, 'raw': raw,
            'size_correct': size_ok, 'grading_correct': grading_ok,
            'valid_json': is_valid, 'time_s': round(elapsed, 2),
        })
        if (i+1) % 10 == 0:
            n = i + 1
            print(f'[{n}/{len(manifest)}] Size: {correct_size}/{n} ({correct_size/n*100:.0f}%) | '
                  f'Grading: {correct_grading}/{n} ({correct_grading/n*100:.0f}%) | '
                  f'JSON: {valid_json}/{n} | Avg: {total_time/n:.1f}s')
    return results, correct_size, correct_grading, valid_json, total_time

print('Helpers ready.')

## Quick Test (3 each from A32, B16, C8 — both modes)

In [ ]:
# Pick 3 images each from A32 (coarse/32mm), B16 (medium/16mm), C8 (fine/8mm)
quick_test_classes = ['A32', 'B16', 'C8']
quick_test = []
for cls in quick_test_classes:
    cls_entries = [e for e in manifest if e['class'] == cls][:3]
    quick_test.extend(cls_entries)
print(f'Quick test: {len(quick_test)} images — {[e["class"] for e in quick_test]}')

for mode in ['zero-shot', 'few-shot']:
    print(f'\n{"=" * 60}')
    print(f'Quick test — {mode}')
    print(f'{"=" * 60}')
    ri = ref_image if mode == 'few-shot' else None
    for entry in quick_test:
        img_path = os.path.join(TEST_DIR, entry['image'])
        raw, elapsed = infer(img_path, mode=mode, ref_img=ri)
        parsed = parse_response(raw)
        gt_size = entry['max_particle_size_mm']
        gt_grading = entry['grading']
        if parsed:
            size_ok = '✓' if parsed.get('max_particle_size_mm') == gt_size else '✗'
            grad_ok = '✓' if parsed.get('grading', '').lower() == gt_grading else '✗'
            verdict = f'Size {size_ok}  Grading {grad_ok}'
        else:
            verdict = 'PARSE FAIL'
        print(f"{entry['class']:>3} | GT: size={gt_size}, grading={gt_grading}")
        print(f"     Pred: {parsed} | {elapsed:.1f}s | {verdict}")
        print()

## Full Benchmark — Zero-Shot
No reference image. True baseline. ~5-10 min.

In [ ]:
print('Running zero-shot benchmark...')
zs_results, zs_size, zs_grading, zs_json, zs_time = run_benchmark(manifest, 'zero-shot')
print(f'\nDone. {len(zs_results)} images in {zs_time:.0f}s')

## Full Benchmark — Few-Shot (with reference image)
Reference chart included in every prompt. ~10-15 min (extra image = more tokens).

In [ ]:
print('Running few-shot benchmark...')
fs_results, fs_size, fs_grading, fs_json, fs_time = run_benchmark(manifest, 'few-shot', ref_img=ref_image)
print(f'\nDone. {len(fs_results)} images in {fs_time:.0f}s')

## Results Comparison

In [ ]:
def summarize(label, results, c_size, c_grading, v_json, t_time):
    n = len(results)
    both = sum(1 for r in results if r['size_correct'] and r['grading_correct'])
    return {
        'label': label, 'n': n,
        'json_pct': round(v_json/n*100, 1),
        'size_pct': round(c_size/n*100, 1),
        'grading_pct': round(c_grading/n*100, 1),
        'both_pct': round(both/n*100, 1),
        'avg_time': round(t_time/n, 2),
    }

zs_summary = summarize('Zero-Shot', zs_results, zs_size, zs_grading, zs_json, zs_time)
fs_summary = summarize('Few-Shot', fs_results, fs_size, fs_grading, fs_json, fs_time)

print(f'{"":<14} {"Zero-Shot":>12} {"Few-Shot":>12} {"Δ":>8}')
print('-' * 48)
for key, label in [('json_pct','JSON Valid %'), ('size_pct','Size Acc %'), ('grading_pct','Grading Acc %'), ('both_pct','Both Correct %'), ('avg_time','Avg Time (s)')]:
    zv = zs_summary[key]; fv = fs_summary[key]
    delta = fv - zv
    sign = '+' if delta > 0 else ''
    print(f'{label:<14} {zv:>12} {fv:>12} {sign}{delta:>7.1f}')

In [ ]:
# Side-by-side bar chart
metrics = ['JSON Valid', 'Size Acc', 'Grading Acc', 'Both Correct']
zs_vals = [zs_summary['json_pct'], zs_summary['size_pct'], zs_summary['grading_pct'], zs_summary['both_pct']]
fs_vals = [fs_summary['json_pct'], fs_summary['size_pct'], fs_summary['grading_pct'], fs_summary['both_pct']]

x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, zs_vals, w, label='Zero-Shot', color='steelblue')
b2 = ax.bar(x + w/2, fs_vals, w, label='Few-Shot', color='coral')
ax.set_ylabel('Accuracy %'); ax.set_title('Zero-Shot vs Few-Shot — Qwen2.5-VL-3B (Base Model)')
ax.set_xticks(x); ax.set_xticklabels(metrics); ax.set_ylim(0, 110)
ax.legend()
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.0f}%', ha='center', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Per-class comparison
def class_breakdown(results):
    by_class = defaultdict(list)
    for r in results: by_class[r['class']].append(r)
    data = {}
    for cls in sorted(by_class):
        cr = by_class[cls]
        data[cls] = {
            'size': sum(1 for r in cr if r['size_correct']) / len(cr) * 100,
            'grading': sum(1 for r in cr if r['grading_correct']) / len(cr) * 100,
        }
    return data

zs_cls = class_breakdown(zs_results)
fs_cls = class_breakdown(fs_results)
classes = sorted(zs_cls.keys())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(classes)); w = 0.35

axes[0].bar(x - w/2, [zs_cls[c]['size'] for c in classes], w, label='Zero-Shot', color='steelblue')
axes[0].bar(x + w/2, [fs_cls[c]['size'] for c in classes], w, label='Few-Shot', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(classes, rotation=45)
axes[0].set_ylabel('Accuracy %'); axes[0].set_title('Size Accuracy by Class'); axes[0].legend(); axes[0].set_ylim(0, 110)

axes[1].bar(x - w/2, [zs_cls[c]['grading'] for c in classes], w, label='Zero-Shot', color='steelblue')
axes[1].bar(x + w/2, [fs_cls[c]['grading'] for c in classes], w, label='Few-Shot', color='coral')
axes[1].set_xticks(x); axes[1].set_xticklabels(classes, rotation=45)
axes[1].set_ylabel('Accuracy %'); axes[1].set_title('Grading Accuracy by Class'); axes[1].legend(); axes[1].set_ylim(0, 110)

plt.suptitle('Per-Class: Zero-Shot vs Few-Shot', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## Error Analysis

In [ ]:
# Show predictions that changed between zero-shot and few-shot
print('Predictions that CHANGED between zero-shot and few-shot:')
print(f'{"Class":>5} {"GT Grading":>12} {"ZS Pred":>10} {"FS Pred":>10} {"ZS→FS":>8}')
print('-' * 50)
changed = 0
for zr, fr in zip(zs_results, fs_results):
    zg = zr['predicted'].get('grading', '?').lower() if zr['predicted'] else '?'
    fg = fr['predicted'].get('grading', '?').lower() if fr['predicted'] else '?'
    if zg != fg:
        gt = zr['gt_grading']
        improved = '✓ fixed' if fg == gt and zg != gt else ('✗ broke' if zg == gt and fg != gt else 'changed')
        print(f"{zr['class']:>5} {gt:>12} {zg:>10} {fg:>10} {improved:>8}")
        changed += 1
print(f'\nTotal changed: {changed}/{len(zs_results)}')

## Save Results

In [ ]:
for label, summary, results in [
    ('zero-shot', zs_summary, zs_results),
    ('few-shot', fs_summary, fs_results),
]:
    fname = f'benchmark_results_{label}.json'
    with open(fname, 'w') as f:
        json.dump({
            'model': 'Qwen2.5-VL-3B-Instruct',
            'mode': label,
            'phase': 'base_model',
            'total_images': summary['n'],
            'json_validity_pct': summary['json_pct'],
            'size_accuracy_pct': summary['size_pct'],
            'grading_accuracy_pct': summary['grading_pct'],
            'both_correct_pct': summary['both_pct'],
            'avg_inference_time_s': summary['avg_time'],
            'results': results,
        }, f, indent=2)
    print(f'Saved {fname}')

print('\nThese files will be compared against Task 4 (post-fine-tuning) results.')
print('Progression: zero-shot → few-shot → fine-tuned (LoRA)')